In [2]:
# # Install Pytorch & other libraries
# %pip install -qqq torch torchvision setuptools scikit-learn

# # Install Hugging Face libraries
# %pip install  --upgrade datasets -qqq accelerate hf-transfer transformers

In [1]:
from datasets import load_dataset

# Dataset id from huggingface.co/dataset
dataset_id = "burtenshaw/PleIAs_common_corpus_code_classification"

# Load raw dataset
dataset = load_dataset(dataset_id)

/home/ollie/.cache/pypoetry/virtualenvs/llm-course-hugging-face-_giC5u1K-py3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(len(dataset["train"]))
print(dataset["train"][0])

127723
{'text': '/*\n * Copyright (c) 2000 Kungliga Tekniska Högskolan\n * (Royal Institute of Technology, Stockholm, Sweden).\n * All rights reserved.\n *\n * Redistribution and use in source and binary forms, with or without\n * modification, are permitted provided that the following conditions\n * are met:\n *\n * 1. Redistributions of source code must retain the above copyright\n *    notice, this list of conditions and the following disclaimer.\n *\n * 2. Redistributions in binary form must reproduce the above copyright\n *    notice, this list of conditions and the following disclaimer in the\n *    documentation and/or other materials provided with the distribution.\n *\n * 3. Neither the name of the Institute nor the names of its contributors\n *    may be used to endorse or promote products derived from this software\n *    without specific prior written permission.\n *\n * THIS SOFTWARE IS PROVIDED BY THE INSTITUTE AND CONTRIBUTORS ``AS IS\'\' AND\n * ANY EXPRESS OR IMPLIED W

In [3]:
from transformers import AutoTokenizer
import os
cores = os.cpu_count()

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize helper function
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")

"""# Tokenize dataset
tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])"""
# Fast Multiprocessing Map
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    num_proc=cores//2,       # <--- THIS IS THE KEY ADDITION
    remove_columns=["text"]
)

tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [4]:
from transformers import AutoModelForSequenceClassification

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Prepare model labels across all splits so test-only labels are included
labels = sorted(set(
    l for split in tokenized_dataset.values() for l in split["labels"]
))
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

# Convert string labels to integer IDs
tokenized_dataset = tokenized_dataset.map(
    lambda batch: {"labels": [label2id[l] for l in batch["labels"]]},
    batched=True,
)

In [5]:
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
Loading weights: 100%|██████████| 136/136 [00:00<00:00, 17960.50it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
import numpy as np
from sklearn.metrics import f1_score

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}


In [7]:
from huggingface_hub import get_token
from transformers import Trainer, TrainingArguments

# Define training args
training_args = TrainingArguments(
    output_dir= "ModernBERT-code-classifier",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True, # bfloat16 training
    optim="adamw_torch_fused", # improved optimizer
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # push to hub parameters
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=get_token(),
    report_to="wandb"
)

# Overfitting

In [ ]:
limited_dataset = tokenized_dataset["train"].select(range(100))

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

# Prevent push thread from hanging
if trainer.push_in_progress is not None:
    trainer.push_in_progress.jobs.cancel()

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ollie/.netrc.
wandb: Currently logged in as: uchavarria (uchavarria-santa-clara-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1
1,No log,3.269058,0.281771
2,No log,2.900238,0.481914
3,No log,2.694979,0.511107
4,No log,2.440569,0.614757
5,No log,2.340088,0.656198


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f0eacd89160>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f0ed0d5f850, execution_count=10 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f0ed0d5f5b0, raw_cell="limited_dataset = tokenized_dataset["train"].selec.." transformed_cell="limited_dataset = tokenized_dataset["train"].selec.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/ollie/Github/llm-course-hugging-face/src/llm_course_hugging_face/chapter3/3_5_Understanding_Learning_Curves.ipynb#X12sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [8]:
# clear memory

import torch
torch.cuda.empty_cache()

del trainer
del model
del limited_dataset

NameError: name 'trainer' is not defined

# Underfitting

In [9]:
import wandb

# define a low learning rate
training_args.learning_rate = 1e-7

limited_dataset = tokenized_dataset["train"].select(range(100))
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

# Prevent push thread from hanging
if trainer.push_in_progress is not None:
    trainer.push_in_progress.jobs.cancel()

wandb.finish()

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 16581.65it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ollie/.netrc.
wandb: Currently logged in as: uchavarria (uchavarria-santa-clara-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1
1,No log,6.066041,0.000118
2,No log,6.050091,0.000118
3,No log,6.038574,0.000117
4,No log,6.031620,0.000117
5,No log,6.029027,0.000118


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]


AttributeError: 'list' object has no attribute 'cancel'

In [10]:
# clear memory

import torch
torch.cuda.empty_cache()

del trainer
del model

# Just right! 🥣

In [11]:
import wandb

# define a valid learning rate
training_args.learning_rate = 5e-5

limited_dataset = tokenized_dataset["train"].select(range(100))
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()


# # Prevent push thread from hanging
# if trainer.push_in_progress is not None:
#     # It's a list, so we loop through it
#     for job in trainer.push_in_progress:
#         job.cancel()

# TRY THIS
if trainer.push_in_progress is not None:
    for job in trainer.push_in_progress.jobs:
        job.cancel()

wandb.finish()

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 15925.22it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,No log,3.142925,0.446663
2,No log,2.428286,0.676104
3,No log,2.091072,0.622565
4,No log,1.973290,0.756544
5,No log,1.906983,0.769055


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]


TypeError: 'PushInProgress' object is not iterable

# Inference

In [1]:
from transformers import pipeline

# load model from huggingface.co/models using our repository id
classifier = pipeline(
    task="text-classification",
    model="argilla/ModernBERT-domain-classifier",
    device=0,
)

sample = """def add_numbers(a, b):
    return a + b"""

classifier(sample)


/home/ollie/.cache/pypoetry/virtualenvs/llm-course-hugging-face-_giC5u1K-py3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 138/138 [00:00<00:00, 20015.01it/s]


[{'label': 'science', 'score': 0.31507962942123413}]